# Tugas 2 Kata Unik Dataset

## 1. Install library terlebih dahulu

In [ ]:
pip install pandas openpyxl numpy scikit-learn

Library Pandas digunakan untuk membaca, mengolah, dan menganalisis data dalam bentuk tabel atau DataFrame. Openpyxl digunakan untuk membaca dan menulis data dalam format Microsoft Excel (.xlsx). NumPy digunakan untuk melakukan operasi numerik serta pengolahan array dan matriks. Sementara itu, Scikit-learn digunakan untuk penerapan metode machine learning, seperti preprocessing data, pembagian data, pemodelan, serta evaluasi hasil klasifikasi atau clustering. Instalasi library tersebut dilakukan agar lingkungan Python memiliki seluruh kebutuhan yang diperlukan dalam proses pengolahan dan analisis dataset.

## 2. Import library

In [ ]:
import pandas as pd
import numpy as np
import re

from scipy.sparse import csr_matrix

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import PCA

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

Library Pandas digunakan untuk membaca dan mengelola dataset dalam bentuk tabel, sedangkan NumPy digunakan untuk melakukan operasi numerik dan pengolahan array. Library re digunakan untuk melakukan pemrosesan teks menggunakan regular expression, seperti membersihkan atau mencari pola tertentu pada teks. SciPy melalui csr_matrix digunakan untuk merepresentasikan data dalam bentuk matriks sparse secara lebih efisien.

Pada tahap pemodelan, train_test_split dari Scikit-learn digunakan untuk membagi dataset menjadi data latih dan data uji. TfidfVectorizer digunakan untuk mengubah data teks menjadi representasi numerik berdasarkan bobot TF-IDF. Selanjutnya, PCA (Principal Component Analysis) digunakan untuk melakukan reduksi dimensi pada data. LogisticRegression digunakan sebagai algoritma klasifikasi. Untuk mengetahui kinerja model, digunakan accuracy_score, classification_report, dan confusion_matrix yang masing-masing digunakan untuk menghitung akurasi, menampilkan metrik evaluasi klasifikasi, serta melihat hasil prediksi dalam bentuk matriks konfusi.

## 3. Membaca dataset

In [ ]:
df = pd.read_excel("dataset_detik_200_berita.xlsx")

df.head()

,id,isi_berita,label
0,1,Pacuan kuda Indonesia mulai diarahkan menuju p...,sport
1,2,"Jelang Asian Games 2026, dukungan untuk 432 at...",sport
2,3,Nova Widianto mulai menangani tim ganda campur...,sport
3,4,Timnas basket putra Indonesia jadi rombongan p...,sport
4,5,"Bagi pelari urban, lari kini bukan lagi sekada...",sport


### Cek jumlah data:

In [ ]:
print("Jumlah data :", len(df))
print("\nNama kolom:")
print(df.columns)

print("\nJumlah data per label:")
print(df["label"].value_counts())

Jumlah data : 200

Nama kolom:
Index(['id', 'isi_berita', 'label'], dtype='object')

Jumlah data per label:
label
sport      100
finance    100
Name: count, dtype: int64


## 4. Mengubah label menjadi numerik

In [ ]:
df["label_num"] = df["label"].map({
    "sport": 1,
    "finance": 0
})

df[["id", "label", "label_num"]].head()

,id,label,label_num
0,1,sport,1
1,2,sport,1
2,3,sport,1
3,4,sport,1
4,5,sport,1


In [ ]:
print(df["label_num"].value_counts())

label_num
1    100
0    100
Name: count, dtype: int64


## 5. Menghitung jumlah kata semua berita

In [ ]:
df["jumlah_kata_asli"] = df["isi_berita"].astype(str).apply(
    lambda x: len(x.split())
)

df[["id", "jumlah_kata_asli"]].head()

total_kata = df["jumlah_kata_asli"].sum()

print("Total seluruh kata :", total_kata)

print(df["jumlah_kata_asli"].describe())

Total seluruh kata : 62652
count     200.000000
mean      313.260000
std       132.649053
min        41.000000
25%       241.250000
50%       293.000000
75%       352.500000
max      1033.000000
Name: jumlah_kata_asli, dtype: float64


### melihat jumlah kata setiap berita

In [ ]:
df[[
    "id",
    "label",
    "jumlah_kata_asli"
]]

,id,label,jumlah_kata_asli
0,1,sport,396
1,2,sport,291
2,3,sport,331
3,4,sport,315
4,5,sport,546
...,...,...,...
195,196,finance,546
196,197,finance,493
197,198,finance,217
198,199,finance,296


## 6. Kamus kata tidak baku

In [ ]:
kamus_tidak_baku = {
    "gak": "tidak",
    "nggak": "tidak",
    "ga": "tidak",
    "enggak": "tidak",
    "yg": "yang",
    "dgn": "dengan",
    "utk": "untuk",
    "krn": "karena",
    "kalo": "kalau",
    "kalok": "kalau",
    "aja": "saja",
    "udah": "sudah",
    "sdh": "sudah",
    "blm": "belum",
    "tdk": "tidak",
    "dr": "dari",
    "dlm": "dalam",
    "jd": "jadi",
    "bgt": "banget",
    "tp": "tetapi",
    "tapi": "tetapi",
    "karna": "karena",
    "trus": "terus",
    "kmrn": "kemarin",
    "dpt": "dapat",
    "hrs": "harus",
    "sm": "sama",
    "sy": "saya"
}

## 7. Kamus bahasa asing

In [ ]:
kamus_asing = {

    # SPORT
    "rider": "pembalap",
    "race": "balapan",
    "racing": "balap",
    "team": "tim",
    "coach": "pelatih",
    "player": "pemain",
    "match": "pertandingan",
    "winner": "pemenang",
    "season": "musim",
    "training": "latihan",
    "game": "pertandingan",
    "games": "pertandingan",
    "manager": "manajer",
    "champion": "juara",
    "championship": "kejuaraan",
    "league": "liga",
    "score": "skor",
    "goal": "gol",
    "final": "final",

    # FINANCE
    "finance": "keuangan",
    "financial": "keuangan",
    "market": "pasar",
    "stock": "saham",
    "stocks": "saham",
    "sale": "penjualan",
    "price": "harga",
    "business": "bisnis",
    "company": "perusahaan",
    "investment": "investasi",
    "investor": "investor",
    "banking": "perbankan",
    "bank": "bank",
    "economy": "ekonomi",
    "economic": "ekonomi",
    "growth": "pertumbuhan",
    "profit": "keuntungan",
    "loss": "kerugian",
    "revenue": "pendapatan"
}

## 8. Fungsi preprocessing

In [ ]:
def preprocessing(text):

    # Pastikan berupa string
    text = str(text)

    # =========================
    # CASE FOLDING
    # =========================
    text = text.lower()


    # =========================
    # HAPUS URL
    # =========================
    text = re.sub(
        r'https?://\S+|www\.\S+',
        ' ',
        text
    )


    # =========================
    # HAPUS EMAIL
    # =========================
    text = re.sub(
        r'\S+@\S+',
        ' ',
        text
    )


    # =========================
    # HAPUS ANGKA
    # =========================
    text = re.sub(
        r'\d+',
        ' ',
        text
    )


    # =========================
    # HAPUS TANDA BACA,
    # SIMBOL DAN EMOTICON
    # =========================

    text = re.sub(
        r'[^a-zA-ZÀ-ÿ\s]',
        ' ',
        text
    )


    # =========================
    # HAPUS SPASI BERLEBIH
    # =========================

    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()


    # =========================
    # TOKENISASI
    # =========================

    kata = text.split()


    hasil = []

    for token in kata:

        # Membakukan kata
        if token in kamus_tidak_baku:
            token = kamus_tidak_baku[token]


        # Bahasa asing -> Indonesia
        if token in kamus_asing:
            token = kamus_asing[token]


        hasil.append(token)


    return " ".join(hasil)

Tahap preprocessing dilakukan untuk membersihkan dan menyeragamkan data teks sebelum digunakan dalam proses pembobotan TF-IDF dan klasifikasi. Proses ini diterapkan melalui fungsi preprocessing(text) yang terdiri dari beberapa tahapan.

Pertama, teks dikonversi menjadi tipe string untuk memastikan data dapat diproses. Selanjutnya dilakukan case folding, yaitu mengubah seluruh huruf menjadi huruf kecil sehingga kata dengan perbedaan kapitalisasi dianggap sebagai kata yang sama.

Tahap berikutnya adalah pembersihan teks, yang meliputi penghapusan URL, alamat email, angka, tanda baca, simbol, dan emoticon yang dianggap tidak diperlukan dalam proses klasifikasi. Setelah itu, spasi berlebih dihapus agar teks menjadi lebih rapi.

Selanjutnya dilakukan tokenisasi dengan memisahkan teks menjadi kata-kata menggunakan fungsi split(). Setiap token kemudian diperiksa menggunakan kamus_tidak_baku untuk mengubah kata tidak baku menjadi bentuk baku. Token juga diperiksa menggunakan kamus_asing untuk mengubah kata berbahasa asing ke dalam bahasa Indonesia. Setelah seluruh proses selesai, token-token tersebut digabungkan kembali menjadi satu teks yang telah bersih dan siap digunakan pada tahap pengolahan berikutnya.

## 9. Terapkan preprocessing ke semua berita

In [ ]:
df["berita_clean"] = df["isi_berita"].apply(preprocessing)

df[[
    "id",
    "isi_berita",
    "berita_clean"
]].head()

,id,isi_berita,berita_clean
0,1,Pacuan kuda Indonesia mulai diarahkan menuju p...,pacuan kuda indonesia mulai diarahkan menuju p...
1,2,"Jelang Asian Games 2026, dukungan untuk 432 at...",jelang asian pertandingan dukungan untuk atlet...
2,3,Nova Widianto mulai menangani tim ganda campur...,nova widianto mulai menangani tim ganda campur...
3,4,Timnas basket putra Indonesia jadi rombongan p...,timnas basket putra indonesia jadi rombongan p...
4,5,"Bagi pelari urban, lari kini bukan lagi sekada...",bagi pelari urban lari kini bukan lagi sekadar...


Setelah fungsi preprocessing dibuat, proses pembersihan teks diterapkan pada seluruh data pada kolom isi_berita menggunakan fungsi apply(preprocessing). Hasil preprocessing kemudian disimpan pada kolom baru bernama berita_clean. Kolom tersebut berisi teks berita yang telah melalui tahapan case folding, pembersihan karakter yang tidak diperlukan, tokenisasi, serta normalisasi kata.

Selanjutnya, df[[...]].head() digunakan untuk menampilkan lima data pertama yang terdiri dari kolom id, isi_berita, dan berita_clean. Tampilan tersebut digunakan untuk melihat perbandingan antara teks berita sebelum dan sesudah dilakukan preprocessing.

## 10. Hitung jumlah kata setelah preprocessing

In [ ]:
df["jumlah_kata_clean"] = df["berita_clean"].apply(
    lambda x: len(x.split())
)

df[[
    "id",
    "jumlah_kata_asli",
    "jumlah_kata_clean"
]].head()

,id,jumlah_kata_asli,jumlah_kata_clean
0,1,396,387
1,2,291,286
2,3,331,325
3,4,315,311
4,5,546,541


Setelah proses preprocessing selesai, dilakukan perhitungan jumlah kata pada teks yang telah dibersihkan. Kolom baru jumlah_kata_clean dibuat dengan menghitung jumlah kata pada setiap data dalam kolom berita_clean. Fungsi split() digunakan untuk memisahkan teks berdasarkan spasi, kemudian len() digunakan untuk menghitung jumlah kata yang dihasilkan.

Selanjutnya, kolom id, jumlah_kata_asli, dan jumlah_kata_clean ditampilkan menggunakan fungsi head() untuk melihat perbandingan jumlah kata sebelum dan sesudah preprocessing pada lima data pertama. Perbandingan ini digunakan untuk mengetahui perubahan jumlah kata akibat proses pembersihan dan normalisasi teks.

### Total:

In [ ]:
print(
    "Total kata sebelum preprocessing :",
    df["jumlah_kata_asli"].sum()
)

print(
    "Total kata setelah preprocessing :",
    df["jumlah_kata_clean"].sum()
)

Total kata sebelum preprocessing : 62652
Total kata setelah preprocessing : 61162


## 11. Ekstrak seluruh kata unik

In [ ]:
semua_kata = []

for berita in df["berita_clean"]:
    semua_kata.extend(
        berita.split()
    )

kata_unik = sorted(
    set(semua_kata)
)

print(
    "Jumlah seluruh kata:",
    len(semua_kata)
)

print(
    "Jumlah kata unik:",
    len(kata_unik)
)

Jumlah seluruh kata: 61162
Jumlah kata unik: 7019


Pada tahap ini dilakukan analisis terhadap kosakata yang terdapat pada seluruh data teks yang telah melalui proses preprocessing. Variabel semua_kata digunakan untuk menampung seluruh kata dari setiap berita. Setiap teks pada kolom berita_clean dipisahkan menjadi kata-kata menggunakan fungsi split(), kemudian seluruh kata digabungkan ke dalam satu list menggunakan extend().

Selanjutnya, set() digunakan untuk menghilangkan kata yang sama sehingga diperoleh kumpulan kata yang unik. Hasil tersebut kemudian diurutkan menggunakan fungsi sorted() dan disimpan dalam variabel kata_unik.

Pada bagian akhir, fungsi len() digunakan untuk menghitung jumlah seluruh kata dan jumlah kata unik. Jumlah seluruh kata menunjukkan total kemunculan kata pada seluruh dokumen, sedangkan jumlah kata unik menunjukkan banyaknya kosakata berbeda yang terdapat dalam dataset setelah preprocessing.

In [ ]:
kata_unik[:100]

['a',
 'aadi',
 'aan',
 'abal',
 'abang',
 'abdi',
 'abdul',
 'abraham',
 'absen',
 'absennya',
 'absorber',
 'abu',
 'abullah',
 'acara',
 'access',
 'account',
 'acd',
 'aceh',
 'acid',
 'acosta',
 'activ',
 'activation',
 'activities',
 'acuan',
 'ada',
 'adain',
 'adalah',
 'adanya',
 'adaptasi',
 'adapun',
 'adhitya',
 'adi',
 'adianto',
 'adik',
 'adil',
 'administrasi',
 'administratif',
 'adna',
 'adopsi',
 'adp',
 'adriatik',
 'ads',
 'adu',
 'aduan',
 'advisory',
 'advokat',
 'aerodinamika',
 'aeronautika',
 'aesi',
 'af',
 'aff',
 'afrianto',
 'afrika',
 'ag',
 'agak',
 'agama',
 'agar',
 'agenda',
 'agent',
 'agraria',
 'agreement',
 'agregat',
 'agresif',
 'agresivitas',
 'agrinas',
 'agung',
 'agus',
 'agusman',
 'agustus',
 'agustusan',
 'ahi',
 'ahmad',
 'ahmed',
 'ahren',
 'ahsanurrohim',
 'ai',
 'aichi',
 'air',
 'airbus',
 'airin',
 'airlangga',
 'airlines',
 'airmen',
 'airnav',
 'airport',
 'airports',
 'ajaib',
 'ajakan',
 'ajang',
 'ajaran',
 'ajir',
 'akademik',

In [ ]:
df_kata_unik = pd.DataFrame({
    "kata_unik": kata_unik
})

df_kata_unik.head(20)

,kata_unik
0,a
1,aadi
2,aan
3,abal
4,abang
5,abdi
6,abdul
7,abraham
8,absen
9,absennya


In [ ]:
df_kata_unik.to_excel(
    "kata_unik.xlsx",
    index=False
)

## 12. Membagi data 160 training dan 40 testing

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(

    df["berita_clean"],
    df["label_num"],

    test_size=40,

    random_state=42,

    stratify=df["label_num"]
)

Pada tahap ini, dataset dibagi menjadi data latih (training data) dan data uji (testing data) menggunakan fungsi train_test_split() dari Scikit-learn. Data yang digunakan sebagai input adalah kolom berita_clean, sedangkan target atau label yang digunakan adalah label_num.

Parameter test_size=40 digunakan untuk menetapkan sebanyak 40 data sebagai data uji, sedangkan data lainnya digunakan sebagai data latih. Parameter random_state=42 digunakan agar proses pembagian data menghasilkan pembagian yang sama setiap kali kode dijalankan. Sementara itu, stratify=df["label_num"] digunakan untuk mempertahankan proporsi masing-masing kelas pada data latih dan data uji.

Hasil pembagian disimpan dalam empat variabel, yaitu X_train_text dan y_train sebagai data teks dan label untuk proses pelatihan, serta X_test_text dan y_test sebagai data teks dan label untuk proses pengujian model.

In [ ]:
print(
    "Jumlah training:",
    len(X_train_text)
)

print(
    "Jumlah testing:",
    len(X_test_text)
)

Jumlah training: 160
Jumlah testing: 40


In [ ]:
print("TRAINING")
print(y_train.value_counts())

print("\nTESTING")
print(y_test.value_counts())

TRAINING
label_num
0    80
1    80
Name: count, dtype: int64

TESTING
label_num
0    20
1    20
Name: count, dtype: int64


## 13. Representasi TF-IDF

In [ ]:
tfidf = TfidfVectorizer(
    min_df=2,
    max_df=0.95
)

X_train_tfidf = tfidf.fit_transform(
    X_train_text
)

X_test_tfidf = tfidf.transform(
    X_test_text
)

In [ ]:
nama_fitur = tfidf.get_feature_names_out()

print(
    "Jumlah fitur TF-IDF:",
    len(nama_fitur)
)

Jumlah fitur TF-IDF: 3038


## 14. Melihat matriks TF-IDF

In [ ]:
tfidf_train_df = pd.DataFrame(
    X_train_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_train_df.head()

,abdul,absen,abu,acara,acd,aceh,acosta,ada,adalah,adanya,...,youtube,yoy,yudha,yudhi,yusrian,zaman,zapp,zona,zulhas,zulkifli
0,0.0,0.0,0.0,0.028231,0.0,0.0,0.0,0.038789,0.014256,0.0,...,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.055677,0.092084,0.0,...,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.144931,0.000000,0.0,...,0.0,0.0,0.0,0.03154,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.021976,0.048461,0.0,...,0.0,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
tfidf_train_df["label"] = y_train.reset_index(
    drop=True
)

tfidf_train_df.head()

,abdul,absen,abu,acara,acd,aceh,acosta,ada,adalah,adanya,...,yoy,yudha,yudhi,yusrian,zaman,zapp,zona,zulhas,zulkifli,label
0,0.0,0.0,0.0,0.028231,0.0,0.0,0.0,0.038789,0.014256,0.0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,0
1,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.055677,0.092084,0.0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,1
2,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.144931,0.000000,0.0,...,0.0,0.0,0.03154,0.0,0.0,0.0,0.0,0.0,0.0,0
3,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,1
4,0.0,0.0,0.0,0.000000,0.0,0.0,0.0,0.021976,0.048461,0.0,...,0.0,0.0,0.00000,0.0,0.0,0.0,0.0,0.0,0.0,1


## 15. Simpan data TF-IDF

In [ ]:
tfidf_train_df.to_excel(
    "data_tfidf_training.xlsx",
    index=False
)

In [ ]:
tfidf_test_df = pd.DataFrame(
    X_test_tfidf.toarray(),
    columns=nama_fitur
)

tfidf_test_df["label"] = y_test.reset_index(
    drop=True
)

tfidf_test_df.to_excel(
    "data_tfidf_testing.xlsx",
    index=False
)

In [ ]:
features = np.array(
    tfidf.get_feature_names_out()
)

jumlah_kelas = y_train.nunique()

class_space_density = np.zeros(
    len(features)
)

y_train_array = y_train.to_numpy()

In [ ]:
for kelas in sorted(
    y_train.unique()
):

    mask = (
        y_train_array == kelas
    )

    X_class = X_train_tfidf[
        mask
    ]


    # Berapa dokumen kelas tersebut
    # mengandung sebuah kata

    document_frequency_class = (
        (X_class > 0)
        .sum(axis=0)
        .A1
    )


    jumlah_dokumen_class = (
        X_class.shape[0]
    )


    class_density = (
        document_frequency_class
        /
        jumlah_dokumen_class
    )


    class_space_density += (
        class_density
    )

In [ ]:
epsilon = 1e-12

icsdf = np.log(

    (jumlah_kelas + epsilon)

    /

    (class_space_density + epsilon)
)

In [ ]:
df_icsdf = pd.DataFrame({

    "kata": features,

    "ICSDF": icsdf
})

df_icsdf.sort_values(
    "ICSDF",
    ascending=False
).head(20)

,kata,ICSDF
10,adaptasi,4.382027
3037,zulkifli,4.382027
3036,zulhas,4.382027
1,absen,4.382027
3033,zaman,4.382027
12,adi,4.382027
3030,yudha,4.382027
5,aceh,4.382027
3013,wti,4.382027
14,adu,4.382027


## 16. TF-IDF × ICSDF

In [ ]:
X_train_icsdf = X_train_tfidf.multiply(
    icsdf
)

X_test_icsdf = X_test_tfidf.multiply(
    icsdf
)

In [ ]:
X_train_icsdf = csr_matrix(
    X_train_icsdf
)

X_test_icsdf = csr_matrix(
    X_test_icsdf
)

## 17. Menentukan kata paling penting

In [ ]:
skor_fitur = np.asarray(
    X_train_icsdf.mean(
        axis=0
    )
).ravel()

In [ ]:
TOP_K = 100

In [ ]:
top_index = np.argsort(
    skor_fitur
)[::-1][:TOP_K]

In [ ]:
kata_penting = features[
    top_index
]

kata_penting[:30]

array(['saham', 'bandara', 'marquez', 'motogp', 'marc', 'balapan',
       'purbaya', 'set', 'aset', 'padel', 'rp', 'aragon', 'olahraga',
       'anggaran', 'emas', 'bezzecchi', 'harga', 'martin', 'atlet',
       'beras', 'saya', 'bbm', 'pertandingan', 'ihsg', 'perdagangan',
       'antonelli', 'masker', 'pasar', 'program', 'triliun'], dtype=object)

In [ ]:
df_kata_penting = pd.DataFrame({

    "kata": kata_penting,

    "skor": skor_fitur[
        top_index
    ]
})

df_kata_penting.head(30)

,kata,skor
0,saham,0.058117
1,bandara,0.058049
2,marquez,0.056400
3,motogp,0.051358
4,marc,0.050835
5,balapan,0.050778
6,purbaya,0.049831
7,set,0.045378
8,aset,0.042438
9,padel,0.039189


## 18. Reduksi menjadi 100 fitur ICSDF

In [ ]:
X_train_selected = X_train_icsdf[
    :,
    top_index
]

X_test_selected = X_test_icsdf[
    :,
    top_index
]

In [ ]:
print(
    "Sebelum seleksi:",
    X_train_tfidf.shape
)

print(
    "Sesudah ICSDF:",
    X_train_selected.shape
)

Sebelum seleksi: (160, 3038)
Sesudah ICSDF: (160, 100)


## 19. Tabel hasil ICSDF

In [ ]:
icsdf_train_df = pd.DataFrame(

    X_train_selected.toarray(),

    columns=kata_penting
)

icsdf_train_df["label"] = (

    y_train.reset_index(
        drop=True
    )
)

icsdf_train_df.head()

,saham,bandara,marquez,motogp,marc,balapan,purbaya,set,aset,padel,...,year,dolar,kerja,level,subsidi,petenis,mereka,aceh,musim,label
0,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.053561,0.0,0.0,0
1,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,1
2,0.0,0.0,0.0,0.0,0.0,0.000000,0.665318,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0
3,0.0,0.0,0.0,0.0,0.0,0.083575,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,1
4,0.0,0.0,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.045516,0.0,0.0,1


In [ ]:
icsdf_train_df.to_excel(
    "data_icsdf_training.xlsx",
    index=False
)

## 20. PCA

In [ ]:
pca = PCA(

    n_components=20,

    random_state=42
)

In [ ]:
X_train_selected_dense = (
    X_train_selected.toarray()
)

X_test_selected_dense = (
    X_test_selected.toarray()
)

In [ ]:
X_train_pca = pca.fit_transform(
    X_train_selected_dense
)

X_test_pca = pca.transform(
    X_test_selected_dense
)

In [ ]:
print(
    X_train_pca.shape
)

print(
    X_test_pca.shape
)

(160, 20)
(40, 20)


## 21. Membuat tabel data reduksi training

In [ ]:
nama_pc = [
    f"PC{i}"
    for i in range(
        1,
        21
    )
]

In [ ]:
df_train_reduksi = pd.DataFrame(

    X_train_pca,

    columns=nama_pc
)

df_train_reduksi["label"] = (

    y_train.reset_index(
        drop=True
    )
)

df_train_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,0.054458,-0.083656,-0.050966,-0.063402,0.049446,0.076339,-0.125293,-0.076448,-0.001681,-0.129239,...,0.059808,-0.047956,0.022836,-0.009557,-0.099971,0.032803,0.155395,-0.237321,0.365373,0
1,0.001516,-0.074960,-0.030823,-0.059307,-0.011138,0.058263,-0.016280,-0.014114,-0.009932,-0.081522,...,0.001136,0.009228,0.020869,0.006865,-0.021050,0.031943,0.095416,-0.066256,-0.004322,1
2,0.037972,-0.092182,-0.058068,-0.061232,0.042450,0.102491,-0.132471,-0.084056,-0.133353,-0.127309,...,0.236267,-0.257832,-0.049353,0.004886,0.010460,-0.035645,-0.075089,0.053086,-0.040341,0
3,-0.014009,-0.091981,-0.046848,-0.079243,0.008004,0.109202,-0.073359,-0.040299,-0.047283,-0.267736,...,-0.473267,-0.058990,-0.311360,-0.104075,0.102788,-0.072941,-0.197435,0.179236,0.086700,1
4,-0.000727,-0.103929,-0.055088,-0.111225,-0.056786,0.099267,0.055226,0.007141,0.008764,-0.088733,...,0.027889,0.052711,0.072312,0.013962,-0.096198,0.057684,0.187838,-0.356487,-0.198698,1


## 22. Data testing

In [ ]:
df_test_reduksi = pd.DataFrame(

    X_test_pca,

    columns=nama_pc
)

df_test_reduksi["label"] = (

    y_test.reset_index(
        drop=True
    )
)

df_test_reduksi.head()

,PC1,PC2,PC3,PC4,PC5,PC6,PC7,PC8,PC9,PC10,...,PC12,PC13,PC14,PC15,PC16,PC17,PC18,PC19,PC20,label
0,0.025333,-0.076968,0.005153,-0.037805,0.010847,0.054619,-0.079845,-0.034686,-0.024508,-0.170149,...,-0.075585,0.103846,0.091166,-0.093016,-0.049398,0.086446,0.739421,0.503766,0.268455,0
1,-0.009996,-0.095285,0.033641,-0.025445,-0.004364,0.060042,-0.031418,-0.004604,-0.012574,-0.124149,...,0.017618,0.092739,0.225136,-0.058508,-0.015073,-0.036555,0.026122,0.045859,0.043395,1
2,-0.098914,0.041605,0.004304,-0.009282,0.005585,0.026705,-0.051870,0.047340,-0.003552,-0.039965,...,0.000947,0.015176,0.067430,-0.005413,-0.010598,-0.012220,0.015563,-0.016303,0.021734,1
3,-0.035422,-0.019355,-0.019390,-0.024950,0.001123,0.031489,-0.027728,-0.020460,-0.010153,-0.063937,...,0.015121,0.016165,0.026433,0.012956,-0.016273,0.021069,0.042084,-0.049683,-0.001718,1
4,0.140381,-0.206879,0.382122,0.168864,0.020674,-0.036795,0.013446,-0.012980,-0.014195,-0.000357,...,-0.067428,0.029574,-0.038317,-0.016361,0.005836,0.014062,0.050209,0.023424,0.013887,0


## 23. Simpan data reduksi

In [ ]:
df_train_reduksi.to_excel(
    "data_reduksi_training.xlsx",
    index=False
)

df_test_reduksi.to_excel(
    "data_reduksi_testing.xlsx",
    index=False
)

## 24. Cek total training dan testing

In [ ]:
print(
    "Training :",
    df_train_reduksi.shape
)

print(
    "Testing :",
    df_test_reduksi.shape
)

Training : (160, 21)
Testing : (40, 21)


## 25. Kolom terakhir harus label

In [ ]:
df_train_reduksi.columns

Index(['PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10',
       'PC11', 'PC12', 'PC13', 'PC14', 'PC15', 'PC16', 'PC17', 'PC18', 'PC19',
       'PC20', 'label'],
      dtype='object')